# CONDOR–FlexDC Behavior Model v2: Prediction, Optimization, and FlexDC Validation

This notebook is the inference companion to:

- `am_unified_model_training_wandb_colab_paths_configured_objective_flexdc_behavior_v2.ipynb`
- `data_center_model_flexdc_behavior_v2.py`
- `am_flexdc_behavior_training_utilities_v2.py`

It loads the new behavior-label checkpoint, predicts raw tracking and per-job QoS, performs multi-start gradient optimization over \(\bar P\), \(R\), and AQA weights, creates a distinct top-k shortlist, and optionally validates every shortlisted candidate in FlexDC.

**Important:** the real FlexDC limits remain p90 ≤ 0.3 and every \(P_j\) ≤ 0.1. Adjustable inference limits such as 0.26 and 0.09 are conservative model-selection margins, not changes to the real system requirements.

## 0. High-level controls

In [ ]:
from pathlib import Path
import os
import sys

RUN_ENV = "colab"  # "colab" or "local"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC.git"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"
FORCE_FRESH_CLONE = True
TORCH_CPU_THREADS = 4  # only used when CUDA is unavailable

# Artifact ZIP acquisition: "upload", "path", or "repo".
CHECKPOINT_SOURCE = "upload"
ARTIFACT_ZIP_PATH = Path("/content/drive/MyDrive/path/condor_set_transformer_old_plus_w2dense_behavior_v2_best_feasibility_artifacts.zip")

USE_WANDB = False
WANDB_MODE = "online"  # online, offline, disabled
WANDB_PROJECT = "flexdc-condor-inference-v2"
WANDB_ENTITY = "amenon06-boston-university"

print("WORKSPACE:", WORKSPACE)
print("CHECKPOINT_SOURCE:", CHECKPOINT_SOURCE)

## 1. Install dependencies

In [ ]:
if RUN_ENV == "colab":
    %pip install -q pandas numpy scipy scikit-learn tqdm matplotlib tabulate wandb
else:
    print("Local mode: using the current environment.")

## 2. Clone or update both repositories

In [ ]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
COMDER_ROOT = WORKSPACE / "comder-main"
FLEXDC_ROOT = WORKSPACE / "flexdc-sim"

if RUN_ENV == "colab":
    if FORCE_FRESH_CLONE:
        !rm -rf "$COMDER_ROOT" "$FLEXDC_ROOT"
    if not COMDER_ROOT.exists():
        !git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" "$COMDER_ROOT"
    if not FLEXDC_ROOT.exists():
        !git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" "$FLEXDC_ROOT"
else:
    # Edit these only in local mode if your repositories live elsewhere.
    COMDER_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/comder-main")
    FLEXDC_ROOT = Path(r"C:/Users/Achuthan Menon/Desktop/Research Work/FlexDC")

print("COMDER_ROOT:", COMDER_ROOT)
print("FLEXDC_ROOT:", FLEXDC_ROOT)

## 3. Resolve paths and verify the new inference files

In [ ]:
TRAIN_DIR = COMDER_ROOT / "am_flexdc" / "train"
INFERENCE_OUTPUT_ROOT = COMDER_ROOT / "am_flexdc" / "results" / "behavior_v2_inference"
ARTIFACT_DIR = COMDER_ROOT / "am_flexdc" / "models" / "behavior_v2_inference_artifacts"
INFERENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

required_code = [
    TRAIN_DIR / "data_center_model_flexdc_behavior_v2.py",
    TRAIN_DIR / "am_flexdc_behavior_training_utilities_v2.py",
    TRAIN_DIR / "am_flexdc_behavior_inference_utilities_v2.py",
    TRAIN_DIR / "am_flexdc_behavior_predict_one_v2.py",
    TRAIN_DIR / "am_flexdc_behavior_optimize_one_v2.py",
    TRAIN_DIR / "am_flexdc_behavior_end_to_end_eval_v2.py",
    TRAIN_DIR / "test_flexdc_behavior_inference_v2.py",
]
missing = [path for path in required_code if not path.exists()]
if missing:
    raise FileNotFoundError("Missing new inference files:\n" + "\n".join(str(path) for path in missing))

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))
print("All inference code files found.")

## 4. Upload or extract the trained-model artifact ZIP

In [ ]:
import shutil
import zipfile

if CHECKPOINT_SOURCE == "upload":
    if RUN_ENV != "colab":
        raise ValueError("upload mode is intended for Colab; use CHECKPOINT_SOURCE='path' locally")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"Upload exactly one artifact ZIP; received {zip_names}")
    source_zip = Path(zip_names[0])
elif CHECKPOINT_SOURCE == "path":
    source_zip = ARTIFACT_ZIP_PATH
    if not source_zip.exists():
        raise FileNotFoundError(source_zip)
elif CHECKPOINT_SOURCE == "repo":
    zip_candidates = sorted((COMDER_ROOT / "am_flexdc" / "models").glob("*behavior_v2*artifacts*.zip"))
    if not zip_candidates:
        raise FileNotFoundError("No behavior-v2 artifact ZIP found under am_flexdc/models")
    source_zip = zip_candidates[-1]
else:
    raise ValueError(CHECKPOINT_SOURCE)

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(source_zip) as archive:
    archive.extractall(ARTIFACT_DIR)

print("Extracted:", source_zip)
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name)

## 5. Select checkpoints and prediction artifacts

In [ ]:
def one_match(pattern):
    matches = sorted(ARTIFACT_DIR.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one match for {pattern}; found {matches}")
    return matches[0]

PRIMARY_CHECKPOINT = one_match("*best_feasibility.pt")
BEST_LOSS_CHECKPOINT = one_match("*best_loss.pt")
BEST_OBJECTIVE_CHECKPOINT = one_match("*best_objective.pt")
HELDOUT_PREDICTIONS_CSV = one_match("*best_feasibility_heldout_predictions.csv")
CHECKPOINT_COMPARISON_CSV = one_match("*checkpoint_comparison.csv")

print("Primary checkpoint:", PRIMARY_CHECKPOINT)
print("Secondary checkpoints:", BEST_LOSS_CHECKPOINT.name, BEST_OBJECTIVE_CHECKPOINT.name)

## 6. Import the behavior-v2 inference utilities

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

from am_flexdc_behavior_inference_utilities_v2 import (
    OptimizationSettings,
    calculate_pr_bounds,
    dataframe_for_csv,
    load_behavior_model,
    margin_calibration_table,
    optimize_candidates,
    predict_configuration,
    read_experiment_config,
    read_workload_config,
    resolve_safety_limits,
    run_flexdc_validation,
    score_candidate_table_with_checkpoint,
    write_json,
)

loaded = load_behavior_model(PRIMARY_CHECKPOINT, device_name="auto")
print("Device:", loaded.device)
print("Checkpoint epoch:", loaded.checkpoint.get("epoch"))
print("Parameters:", loaded.model.parameter_count())
print("Exact thresholds:", loaded.constants.tracking_threshold, loaded.constants.qos_threshold)

if not torch.cuda.is_available():
    torch.set_num_threads(TORCH_CPU_THREADS)
    print("CPU torch threads:", torch.get_num_threads())


## 7. Run structural tests before inference

In [ ]:
# Run lightweight structural tests before the full optimizer cell.
# T2 already verifies finite gradients from the model outputs back to Pbar, R,
# and the weight inputs. The previous version also ran a second full backward
# pass inside the test subprocess; that was redundant and could be slow or
# memory-heavy in Colab.

import subprocess

DEFAULT_TEST_WORKLOAD = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
DEFAULT_TEST_EXPERIMENT = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"
DEFAULT_GRADIENT_CONFIG = FLEXDC_ROOT / "configs" / "gradient_descent" / "gradient_descent_flexdc_paper_objective.ini"
DEFAULT_CLUSTER_CONFIG = FLEXDC_ROOT / "configs" / "cluster" / "cluster.ini"

command = [
    sys.executable,
    "-u",
    str(TRAIN_DIR / "test_flexdc_behavior_inference_v2.py"),
    "--checkpoint", str(PRIMARY_CHECKPOINT),
    "--workload-config", str(DEFAULT_TEST_WORKLOAD),
    "--experiment-config", str(DEFAULT_TEST_EXPERIMENT),
    "--flexdc-root", str(FLEXDC_ROOT),
    "--gradient-config", str(DEFAULT_GRADIENT_CONFIG),
    "--cluster-config", str(DEFAULT_CLUSTER_CONFIG),
    "--device", "auto",
]

print("Running structural tests:")
print(" ".join(f'"{part}"' if " " in part else part for part in command))

completed = subprocess.run(
    command,
    cwd=TRAIN_DIR,
    text=True,
    capture_output=True,
)

print("\nSTDOUT:\n", completed.stdout)
if completed.stderr.strip():
    print("\nSTDERR:\n", completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        f"Inference structural tests failed with return code {completed.returncode}. "
        "See STDOUT/STDERR above."
    )

print("Structural tests passed. The real multi-start optimizer is tested in the optimization cell below.")

## 8. Choose the scenario and adjustable safety/optimization controls

In [ ]:
# Default example: W2-LU, 1000 servers, utilization 0.6, traditional ISO hour 16.
WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"
GRADIENT_CONFIG = FLEXDC_ROOT / "configs" / "gradient_descent" / "gradient_descent_flexdc_paper_objective.ini"
CLUSTER_CONFIG = FLEXDC_ROOT / "configs" / "cluster" / "cluster.ini"
SERVER_COUNT_OVERRIDE = None
UTILIZATION_OVERRIDE = 0.6

START_PBAR = 0.472128
START_R = 0.102206
START_WEIGHTS = [0.258019617529, 0.250894690209, 0.252694830182, 0.23839086208]

# Adjustable model-selection limits. Set either absolute limits or margins.
TRACKING_LIMIT = 0.26  # use None to derive 0.3 - TRACKING_MARGIN
QOS_LIMIT = 0.09      # use None to derive 0.1 - QOS_MARGIN
TRACKING_MARGIN = 0.04
QOS_MARGIN = 0.01

OPTIMIZATION_MODE = "margin_constrained"  # pure_objective, exact_constrained, margin_constrained
MULTI_STARTS = 512
OPTIMIZATION_ITERATIONS = 1500
OPTIMIZATION_LR = 0.03
OPTIMIZATION_MIN_LR = 5e-4
TOP_K = 5
TRACKING_PENALTY = 2000.0
QOS_PENALTY = 2000.0
PENALTY_RAMP_FRACTION = 0.30
NEAR_EQUAL_START_FRACTION = 0.25
HIGH_P_LOW_R_START_FRACTION = 0.25
CANDIDATE_DISTANCE = 0.03
RANDOM_SEED = 0

# Optional trust-region controls. None means no extra restriction beyond the
# physical FlexDC P/R polytope and positive sum-to-one weights.
WEIGHT_MIN = None
WEIGHT_MAX = None
R_OVER_P_MAX = None  # e.g. 0.6 only after confirming this experiment choice
PBAR_MIN_OVERRIDE = None
PBAR_MAX_OVERRIDE = None
R_MIN_OVERRIDE = None
R_MAX_OVERRIDE = None

# Actual FlexDC validation is slow. Inspect model-only results first, then set True.
RUN_FLEXDC_VALIDATION = False
VALIDATE_STARTING_POINT = True
RUN_NAME = "W2_LU_behavior_v2"
RUN_DIR = INFERENCE_OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

## 9. Inspect the scenario, physical bounds, and safety limits

In [ ]:
workload = read_workload_config(WORKLOAD_CONFIG)
experiment = read_experiment_config(
    EXPERIMENT_CONFIG,
    server_count_override=SERVER_COUNT_OVERRIDE,
    utilization_override=UTILIZATION_OVERRIDE,
)
bounds = calculate_pr_bounds(workload)
safety = resolve_safety_limits(
    loaded.constants,
    tracking_limit=TRACKING_LIMIT,
    qos_limit=QOS_LIMIT,
    tracking_margin=TRACKING_MARGIN,
    qos_margin=QOS_MARGIN,
)

scenario_table = pd.DataFrame([
    {"Setting": "Workload", "Value": Path(WORKLOAD_CONFIG).name},
    {"Setting": "Job types J", "Value": workload.job_count},
    {"Setting": "Server count", "Value": experiment.server_count},
    {"Setting": "Utilization", "Value": experiment.utilization},
    {"Setting": "Pbar range", "Value": f"{bounds.pbar_lower_kw_per_server:.4f}–{bounds.pbar_upper_kw_per_server:.4f}"},
    {"Setting": "Combined Pbar+R limit", "Value": bounds.pr_upper_kw_per_server},
    {"Setting": "Exact p90 threshold", "Value": safety.exact_tracking_threshold},
    {"Setting": "Selection p90 limit", "Value": safety.selection_tracking_limit},
    {"Setting": "Exact QoS threshold", "Value": safety.exact_qos_threshold},
    {"Setting": "Selection QoS limit", "Value": safety.selection_qos_limit},
])
display(scenario_table.style.hide(axis="index"))

## 10. Calibrate candidate-selection margins from heldout predictions

In [ ]:
calibration = margin_calibration_table(HELDOUT_PREDICTIONS_CSV)
calibration.to_csv(RUN_DIR / "margin_calibration.csv", index=False)

# Show the exact combination selected above and several low-false-positive alternatives.
selected_row = calibration[
    np.isclose(calibration["Tracking_Limit"], safety.selection_tracking_limit)
    & np.isclose(calibration["QoS_Limit"], safety.selection_qos_limit)
]
display(Markdown("### Selected limits"))
display(selected_row.style.hide(axis="index").format(precision=4))
display(Markdown("### Conservative alternatives"))
display(calibration.head(12).style.hide(axis="index").format(precision=4))

## 11. Predict the starting configuration

In [ ]:
start_prediction, start_per_job = predict_configuration(
    loaded,
    workload=workload,
    experiment=experiment,
    pbar_kw_per_server=START_PBAR,
    r_kw_per_server=START_R,
    weights=START_WEIGHTS,
    safety=safety,
    bounds=bounds,
    r_over_p_max=R_OVER_P_MAX,
)

start_table = pd.DataFrame([{
    "Configuration": "Starting configuration",
    "Pbar": start_prediction["Pbar_kw_per_server"],
    "R": start_prediction["R_kw_per_server"],
    "Weights": str([round(x, 4) for x in start_prediction["weights"]]),
    "Pred mean tracking": start_prediction["Predicted_Mean_Tracking"],
    "Pred p90": start_prediction["Predicted_P90_Tracking"],
    "Pred max Pj": start_prediction["Predicted_Max_Pj"],
    "Pred M_RSR": start_prediction["Predicted_M_RSR"],
    "Pred objective": start_prediction["Predicted_Full_Objective"],
    "Exact pass": start_prediction["Exact_Both_Pass"],
    "Safety pass": start_prediction["Safety_Both_Pass"],
}])
display(start_table.style.hide(axis="index").format(precision=5))
display(Markdown("### Per-job QoS prediction"))
display(start_per_job.style.hide(axis="index").format(precision=5))
write_json(RUN_DIR / "starting_prediction.json", {"summary": start_prediction, "per_job": start_per_job.to_dict(orient="records")})

## 12. Run multi-start gradient optimization

In [ ]:
settings = OptimizationSettings(
    starts=MULTI_STARTS,
    iterations=OPTIMIZATION_ITERATIONS,
    learning_rate=OPTIMIZATION_LR,
    minimum_learning_rate=OPTIMIZATION_MIN_LR,
    mode=OPTIMIZATION_MODE,
    tracking_penalty=TRACKING_PENALTY,
    qos_penalty=QOS_PENALTY,
    penalty_ramp_fraction=PENALTY_RAMP_FRACTION,
    top_k=TOP_K,
    candidate_distance=CANDIDATE_DISTANCE,
    random_seed=RANDOM_SEED,
    near_equal_start_fraction=NEAR_EQUAL_START_FRACTION,
    high_p_low_r_start_fraction=HIGH_P_LOW_R_START_FRACTION,
    weight_min=WEIGHT_MIN,
    weight_max=WEIGHT_MAX,
    r_over_p_max=R_OVER_P_MAX,
    pbar_min_override=PBAR_MIN_OVERRIDE,
    pbar_max_override=PBAR_MAX_OVERRIDE,
    r_min_override=R_MIN_OVERRIDE,
    r_max_override=R_MAX_OVERRIDE,
)

all_candidates, top_k_candidates, trajectory = optimize_candidates(
    loaded,
    workload=workload,
    experiment=experiment,
    bounds=bounds,
    safety=safety,
    settings=settings,
    initial_pbar=START_PBAR,
    initial_reserve=START_R,
    initial_weights=START_WEIGHTS,
)

# Cross-score the shortlist with the best-loss and best-objective checkpoints.
if len(top_k_candidates):
    top_k_candidates = score_candidate_table_with_checkpoint(
        top_k_candidates,
        checkpoint_path=BEST_LOSS_CHECKPOINT,
        workload=workload,
        experiment=experiment,
        safety=safety,
        prefix="BestLoss",
    )
    top_k_candidates = score_candidate_table_with_checkpoint(
        top_k_candidates,
        checkpoint_path=BEST_OBJECTIVE_CHECKPOINT,
        workload=workload,
        experiment=experiment,
        safety=safety,
        prefix="BestObjective",
    )

for name, frame in [
    ("all_starts", all_candidates),
    ("top_k_predicted", top_k_candidates),
    ("trajectory", trajectory),
]:
    dataframe_for_csv(frame).to_csv(RUN_DIR / f"{name}.csv", index=False)

print("Exact-feasible final starts:", int(all_candidates["Exact_Both_Pass"].sum()))
print("Safety-feasible final starts:", int(all_candidates["Safety_Both_Pass"].sum()))
print("Distinct top-k candidates:", len(top_k_candidates))
if top_k_candidates.empty:
    print("No candidate met the configured selection limits. No unconstrained fallback was silently substituted.")

## 13. Inspect the optimization trajectory and top-k shortlist

In [ ]:
import matplotlib.pyplot as plt

if len(trajectory):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(trajectory["Iteration"], trajectory["Best_Predicted_Objective"], label="Best predicted objective")
    ax.set_xlabel("Optimization iteration")
    ax.set_ylabel("Predicted full objective")
    ax.set_title("Model-only multi-start optimization")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

if len(top_k_candidates):
    display_columns = [
        "Candidate_Rank", "Pbar_kw_per_server", "R_kw_per_server", "weights",
        "Predicted_P90_Tracking", "Predicted_Max_Pj", "Predicted_M_RSR",
        "Predicted_Full_Objective", "Safety_Tracking_Slack", "Safety_QoS_Slack",
        "BestLoss_Safety_Pass", "BestObjective_Safety_Pass",
    ]
    display(top_k_candidates[[c for c in display_columns if c in top_k_candidates]].style.hide(axis="index").format(precision=5))

## 14. Optionally validate the starting point and top-k candidates in FlexDC

In [ ]:
validation_rows = []
per_job_validation_rows = []

if RUN_FLEXDC_VALIDATION:
    validation_specs = []
    if VALIDATE_STARTING_POINT:
        validation_specs.append(("start", 0, START_PBAR, START_R, START_WEIGHTS, start_prediction))
    for _, row in top_k_candidates.iterrows():
        weights = row["weights"] if isinstance(row["weights"], list) else json.loads(row["weights"])
        validation_specs.append((f"rank_{int(row['Candidate_Rank'])}", int(row["Candidate_Rank"]), float(row["Pbar_kw_per_server"]), float(row["R_kw_per_server"]), weights, row.to_dict()))

    for label, rank, pbar, reserve, weights, predicted in validation_specs:
        actual, actual_jobs = run_flexdc_validation(
            python_executable=sys.executable,
            flexdc_root=FLEXDC_ROOT,
            gradient_config=GRADIENT_CONFIG,
            experiment_config=EXPERIMENT_CONFIG,
            cluster_config=CLUSTER_CONFIG,
            workload_config=WORKLOAD_CONFIG,
            output_label=f"{RUN_NAME}_{label}",
            pbar_kw_per_server=pbar,
            r_kw_per_server=reserve,
            weights=weights,
            utilization=experiment.utilization,
            constants=loaded.constants,
            timeout_seconds=1800,
        )
        validation_rows.append({
            "Candidate": label,
            "Rank": rank,
            "Pbar": pbar,
            "R": reserve,
            "Weights": weights,
            "Pred p90": predicted["Predicted_P90_Tracking"],
            "Actual p90": actual["Actual_P90_Tracking"],
            "Pred max Pj": predicted["Predicted_Max_Pj"],
            "Actual max Pj": actual["Actual_Max_Pj"],
            "Pred M_RSR": predicted["Predicted_M_RSR"],
            "Actual M_RSR": actual["Actual_M_RSR"],
            "Pred objective": predicted["Predicted_Full_Objective"],
            "Actual objective": actual["Actual_Full_Objective"],
            "Pred safety pass": predicted["Safety_Both_Pass"],
            "Actual pass": actual["Actual_Both_Pass"],
            "FlexDC output": actual["FlexDC_Output_Dir"],
        })
        jobs = actual_jobs.copy()
        jobs.insert(0, "Candidate", label)
        jobs["Job_Type"] = workload.job_names
        predicted_probs = np.asarray(predicted["Predicted_QoS_Probabilities"], dtype=float)
        jobs["Predicted_Pj"] = predicted_probs
        jobs["Pj_Error"] = predicted_probs - jobs["Actual_Pj"]
        per_job_validation_rows.append(jobs)

    validation_df = pd.DataFrame(validation_rows)
    per_job_validation_df = pd.concat(per_job_validation_rows, ignore_index=True)
    dataframe_for_csv(validation_df).to_csv(RUN_DIR / "predicted_vs_actual.csv", index=False)
    dataframe_for_csv(per_job_validation_df).to_csv(RUN_DIR / "per_job_predicted_vs_actual.csv", index=False)
    display(validation_df.style.hide(axis="index").format(precision=5))

    feasible = validation_df[validation_df["Actual pass"]]
    if len(feasible):
        selected_actual = feasible.sort_values("Actual objective").iloc[0]
        display(Markdown("### Selected actual-feasible candidate"))
        display(pd.DataFrame([selected_actual]).style.hide(axis="index").format(precision=5))
    else:
        print("No FlexDC-validated candidate passed both real constraints.")
else:
    print("FlexDC validation is disabled. Review the model-only shortlist, then set RUN_FLEXDC_VALIDATION=True.")

## 15. Optional W&B logging for inference

In [ ]:
if USE_WANDB and WANDB_MODE != "disabled":
    import wandb
    os.environ["WANDB_MODE"] = WANDB_MODE
    if WANDB_MODE == "online":
        wandb.login(relogin=True, verify=True)
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=RUN_NAME,
        mode=WANDB_MODE,
        config={
            "checkpoint": str(PRIMARY_CHECKPOINT),
            "workload": Path(WORKLOAD_CONFIG).name,
            "server_count": experiment.server_count,
            "utilization": experiment.utilization,
            "tracking_limit": safety.selection_tracking_limit,
            "qos_limit": safety.selection_qos_limit,
            **settings.__dict__,
        },
    )
    run.log({
        "optimization/exact_feasible_starts": int(all_candidates["Exact_Both_Pass"].sum()),
        "optimization/safety_feasible_starts": int(all_candidates["Safety_Both_Pass"].sum()),
        "optimization/top_k_count": len(top_k_candidates),
    })
    if len(trajectory):
        run.log({"optimization/trajectory": wandb.Table(dataframe=trajectory)})
    if len(top_k_candidates):
        run.log({"optimization/top_k": wandb.Table(dataframe=dataframe_for_csv(top_k_candidates))})
    if RUN_FLEXDC_VALIDATION and len(validation_rows):
        run.log({"validation/predicted_vs_actual": wandb.Table(dataframe=dataframe_for_csv(validation_df))})
        run.summary["validation/actual_feasible_count"] = int(validation_df["Actual pass"].sum())
    run.finish()
else:
    print("W&B inference logging disabled.")

## 16. Package and download all outputs

In [ ]:
import shutil

summary = {
    "checkpoint": str(PRIMARY_CHECKPOINT),
    "checkpoint_epoch": int(loaded.checkpoint.get("epoch", -1)),
    "workload_config": str(WORKLOAD_CONFIG),
    "experiment_config": str(EXPERIMENT_CONFIG),
    "bounds": bounds.to_dict(),
    "safety": safety.to_dict(),
    "settings": settings.__dict__,
    "exact_feasible_starts": int(all_candidates["Exact_Both_Pass"].sum()),
    "safety_feasible_starts": int(all_candidates["Safety_Both_Pass"].sum()),
    "top_k_count": int(len(top_k_candidates)),
    "flexdc_validation_ran": bool(RUN_FLEXDC_VALIDATION),
}
write_json(RUN_DIR / "run_summary.json", summary)
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Packaged:", zip_path)

if RUN_ENV == "colab":
    from google.colab import files
    files.download(zip_path)